# Exploratory analysis — European power fundamentals

This notebook walks through the analysis behind the dashboard, step by step:

1. Load and validate hourly data for one bidding zone.
2. Check how much of the price residual demand explains.
3. Look at the forecast-error structure.
4. Characterise spikes and negative prices.
5. Run the transparent signal through the honest backtest.
6. Generate the rule-based daily brief.

> **Data:** without an `ENTSOE_API_TOKEN` this uses the **synthetic sample** in `data/sample/`.
> The relationships in that data come from the generator (see `data/sample/README.md`). They are not discoveries about real markets.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.data_loader import load_market_data
from src.feature_engineering import build_features, daily_summary, flag_spikes
from src.signal_engine import build_signal_frame, component_diagnostics
from src.backtester import run_backtest
from src.market_commentary import generate_daily_brief
from src.config import SignalParams, BacktestParams

pd.set_option("display.precision", 2)
ZONE, START, END = "DE_LU", "2024-01-01", "2024-06-30"
data = load_market_data(ZONE, START, END, warmup_days=0)
print(data.source, "|", data.notices[0][:80], "...")
data.quality.to_frame()

sample | SYNTHETIC SAMPLE DATA: generated by scripts/generate_sample_data.py to mimic ENT ...


,missing (raw),interpolated,still missing,longest gap (h),coverage
column,,,,,
load_actual,0,0,0,0,100.0%
load_forecast,0,0,0,0,100.0%
price,0,0,0,0,100.0%
solar_actual,0,0,0,0,100.0%
solar_forecast,0,0,0,0,100.0%
wind_actual,0,0,0,0,100.0%
wind_forecast,0,0,0,0,100.0%


## 1. Feature set

`build_features` adds residual demand, forecast errors (actual − forecast), calendar flags and price changes. It also adds *trailing* z-scores and volatility regimes, which only ever look backwards.

In [2]:
feat = build_features(data.frame)
feat[["price", "load_actual", "wind_actual", "solar_actual", "residual_demand",
      "residual_demand_error", "price_change_24h", "vol_regime"]].describe().T

,count,mean,std,min,25%,50%,75%,max
price,4367.0,67.29,36.71,-86.66,36.22,79.82,93.44,167.99
load_actual,4367.0,50900.40,8173.11,34519.00,43610.00,50665.00,57862.50,69468.00
wind_actual,4367.0,17487.05,15020.92,129.00,5701.00,12194.00,26733.00,59574.00
solar_actual,4367.0,7656.96,11662.77,0.00,0.00,551.00,11647.50,55939.00
residual_demand,4367.0,25756.38,15874.34,-26230.00,14604.50,28405.00,37400.00,60988.00
residual_demand_error,4367.0,-518.14,2988.49,-12518.00,-2076.50,-296.00,1204.00,10718.00
price_change_24h,4342.0,-0.31,30.79,-108.06,-13.57,-0.82,12.38,151.20


## 2. Residual demand as the price driver

Wind and solar have near-zero marginal cost, so the dispatchable fleet has to cover the *residual*.
The higher that residual sits on the merit order, the more expensive the marginal plant. A convex (quadratic) fit captures the steepening at the top of the stack.

In [3]:
x = feat["residual_demand"] / 1000
coef = np.polyfit(x, feat["price"], 2)
pred = np.polyval(coef, x)
r2 = 1 - ((feat["price"] - pred) ** 2).sum() / ((feat["price"] - feat["price"].mean()) ** 2).sum()
print(f"corr(residual demand, price) = {feat['residual_demand'].corr(feat['price']):.2f}")
print(f"quadratic R^2                = {r2:.2f}")

buckets = pd.qcut(feat["residual_demand"] / 1000, 10)
feat.groupby(buckets, observed=True)["price"].describe()[["count", "25%", "50%", "75%"]]

corr(residual demand, price) = 0.90
quadratic R^2                = 0.82


,count,25%,50%,75%
residual_demand,,,,
"(-26.231, 3.042]",437.0,-0.77,2.75,5.79
"(3.042, 11.489]",437.0,12.36,19.96,29.53
"(11.489, 17.991]",436.0,25.68,35.84,53.55
"(17.991, 23.924]",437.0,44.22,68.43,80.74
"(23.924, 28.405]",437.0,71.79,80.74,87.89
"(28.405, 32.086]",436.0,78.51,84.70,91.93
"(32.086, 35.738]",437.0,79.83,87.70,95.33
"(35.738, 39.09]",436.0,82.51,92.31,99.60
"(39.09, 44.458]",437.0,87.39,94.90,103.91


## 3. Forecast errors

All errors are `actual − day-ahead forecast`. A positive *residual-demand error* means the system was shorter than the auction assumed.

In [4]:
errs = feat[["load_error", "wind_error", "solar_error", "residual_demand_error"]] / 1000
summary = pd.DataFrame({"bias (GW)": errs.mean(), "MAE (GW)": errs.abs().mean(), "std (GW)": errs.std(),
                        "lag-1h autocorr": errs.apply(lambda s: s.autocorr(1)),
                        "lag-24h autocorr": errs.apply(lambda s: s.autocorr(24))})
summary

,bias (GW),MAE (GW),std (GW),lag-1h autocorr,lag-24h autocorr
load_error,-0.02,0.50,0.63,0.91,0.39
wind_error,0.48,1.90,2.65,0.92,0.29
solar_error,0.02,0.60,1.26,0.89,0.15
residual_demand_error,-0.52,2.24,2.99,0.91,0.29


In [5]:
# Wind errors by hour of day: when is the forecast least reliable?
(feat.groupby("hour")["wind_error"].agg(lambda s: s.abs().mean()) / 1000).round(2).to_frame("wind MAE (GW)").T

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
wind MAE (GW),1.86,1.8,1.8,1.84,1.75,1.8,1.84,1.77,1.71,1.81,...,1.93,1.98,2.09,2.17,2.08,2.12,2.05,1.95,1.88,1.86


## 4. Spikes and negative prices

In [6]:
spike, level = flag_spikes(feat["price"], percentile=0.9)
groups = np.select([spike, feat["is_negative"]], ["Spike", "Negative"], "Normal")
cmp = feat.groupby(groups)[["price", "residual_demand", "wind_actual", "solar_actual", "load_error"]].mean()
cmp[["residual_demand", "wind_actual", "solar_actual", "load_error"]] /= 1000
print(f"Spike threshold (90th percentile): {level:.1f} EUR/MWh")
cmp

Spike threshold (90th percentile): 104.5 EUR/MWh


,price,residual_demand,wind_actual,solar_actual,load_error
Negative,-17.21,-11.55,38.75,20.75,5.40e-02
Normal,64.71,24.91,18.01,7.73,-1.52e-03
Spike,114.98,44.24,6.61,3.08,-1.75e-01


In [7]:
daily = daily_summary(feat)
monthly = daily.groupby(daily.index.to_period("M")).agg(baseload=("baseload", "mean"), negative_hours=("negative_hours", "sum"),
                                                       avg_range=("intraday_range", "mean"))
monthly

,baseload,negative_hours,avg_range
date,,,
2024-01,79.14,26,65.49
2024-02,54.71,32,68.14
2024-03,57.62,24,60.53
2024-04,66.92,14,69.59
2024-05,68.95,24,100.69
2024-06,75.85,10,76.08


## 5. Signal + honest backtest

The decision for delivery day D is taken on D-1, before the 12:00 CET gate. The inputs are:

* the day-ahead forecasts for D;
* actuals up to D-2;
* prices up to D-1.

The target is the price change against the same hour on D-1. Costs are charged on every MWh.

In [8]:
sig = build_signal_frame(feat, SignalParams(horizon="hourly"))
res = run_backtest(sig, BacktestParams(cost_per_mwh=1.0))
pd.Series(res.metrics).to_frame("value")

,value
eligible_rows,4342.00
n_trades,893.00
trade_frequency,0.21
n_days,181.00
hit_rate,0.71
hit_rate_long,0.70
hit_rate_short,0.71
base_rate_up,0.48
n_long,316.00
n_short,577.00


In [9]:
component_diagnostics(sig)

,active observations,hit rate,avg. signed move (EUR/MWh)
component,,,
Residual demand,1165,0.99,42.63
Load surprise,2686,0.48,-3.05
Renewable surprise,2135,0.51,-1.07
Combined score,2765,0.62,7.99


**How to read this.**

* The residual-demand vote is right most of the time. Day-ahead prices are largely set by forecast residual demand, and the market knows that forecast too. Beating yesterday's price is therefore **not** the same as beating the pre-auction market.
* The lagged surprise votes are roughly coin flips. Forecast errors matter for intraday and imbalance prices on the day, not for the next auction.

That is a useful negative result.

## 6. Rule-based daily brief

In [10]:
print(generate_daily_brief(feat, ZONE, "2024-04-14", data.zone_prices, data.flows).to_markdown())

# Daily Power Market Brief — Germany/Luxembourg — 2024-04-14
**Balanced system despite low wind: baseload €54/MWh (−€4 d/d)**

## 1. Price action
Germany/Luxembourg day-ahead baseload averaged €54/MWh, down €4 on the previous day and €6 below its 30-day average. As a weekend day there was no standard peak block; demand-driven shape was flatter than on weekdays. The daily high of €91/MWh printed at 19:00 (evening peak), while the low of €5/MWh came at 11:00 (midday).

## 2. Demand and residual load
Residual demand averaged 22.1 GW, 3.3 GW below its 30-day average (−13%), mainly driven by softer load and stronger solar. Overall, system tightness was close to recent norms. The residual-load peak of 40.6 GW came at 20:00. The zone was a net importer on the day (average 1.9 GW), with imports helping to relieve local tightness. The widest average spread was against FR (−€27/MWh FR minus DE_LU), pointing to congested interconnection on that border for part of the day.

## 3. Renewables
Wind a